# Executed bounded mushroom AutoML comparison

**Contribution owner: Muneeb Shakoor - integration, baseline reproduction, AutoML and error analysis (tasks 13, 15, 18, 24).**
Builds on Viviana Pajic's Mushroom exploration/evaluation (tasks 8/11) and the shared
AI-assisted starter maintained by Tomislav Novosel. Codex implemented and executed
this work at Muneeb's request on 8-9 October 2026. Muneeb selected the scope and
preservation requirements; this does not claim independent human authorship or
completed human code review. See [AI assistance](../docs/AI_USE.md) and
[reproduction and decisions](../docs/MUSHROOM_WORKFLOW.md).

Use Python 3.11 with `requirements-pycaret.lock.txt` for the recorded environment.


This replaces the unexecuted template. PyCaret compares four complete sklearn
pipelines: LR, RF, Extra Trees and histogram gradient boosting. Extra Trees tests
stronger split randomization; boosting tests sequential correction of errors.
The shortlist is bounded to practical CPU models; no deep learning or broad search
is justified by 4,000 development rows at this checkpoint.


## Two clearly separated stages

**PyCaret screening:** original 3,000 training rows; original 1,000 validation rows
serve only as PyCaret's administrative holdout and are not scored. The reserved
test is never passed to PyCaret. Screening uses five stratified folds repeated
twice and ranks custom average precision, not accuracy or recall alone.

Native PyCaret preprocessing is disabled. Each candidate is a complete shared
pipeline, so imputation/encoding/scaling are fitted inside each CV fold. PyCaret
may downcast numeric dtypes during setup; this is another reason to distinguish
screening from the common evaluation.

**Common evaluation:** evaluate the two new families on exactly the same 4,000-row
folds, feature contract and environment as notebook 02. Combine these with the
verified baseline results. These scores support the fair candidate comparison.
No threshold is optimized, and this is not hyperparameter tuning.


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'src/mushrooms.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open the notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import json
import pandas as pd
from mushroom_workflow import REPORTS, PROCESSED
from mushroom_automl import run_automl
screening, comparison = run_automl()
display(screening[['model', 'AP', 'AUC', 'Recall', 'Prec.', 'F1', 'TT (Sec)']])


PyCaret screening: four complete pipelines, 3,000 rows, 5 folds x 2 repeats.


                 model       AP      AUC   Recall    Prec.       F1
         random_forest 0.763011 0.823890 0.496485 0.832395 0.621366
hist_gradient_boosting 0.749369 0.813593 0.541823 0.784431 0.640235
           extra_trees 0.739524 0.795718 0.479342 0.812216 0.602141
   logistic_regression 0.624164 0.693998 0.353870 0.691686 0.467770


Now evaluate the two additional families on the same 4,000-row folds as the baselines.


Evaluating extra_trees: 5 folds x 2 repeats...


  AP=0.754268, ROC-AUC=0.804034


Evaluating hist_gradient_boosting: 5 folds x 2 repeats...


  AP=0.758460, ROC-AUC=0.812080


                 model  cv_average_precision  cv_roc_auc  oof_poison_recall
         random_forest              0.783075    0.833420           0.520792
hist_gradient_boosting              0.758460    0.812080           0.536634
           extra_trees              0.754268    0.804034           0.482508
   logistic_regression              0.630651    0.689445           0.345875
     majority_baseline              0.378750    0.500000           0.000000


,model,AP,AUC,Recall,Prec.,F1,TT (Sec)
1,random_forest,0.763011,0.823890,0.496485,0.832395,0.621366,0.428
3,hist_gradient_boosting,0.749369,0.813593,0.541823,0.784431,0.640235,0.300
2,extra_trees,0.739524,0.795718,0.479342,0.812216,0.602141,0.456
0,logistic_regression,0.624164,0.693998,0.353870,0.691686,0.467770,0.059


## Common-protocol evidence

The following table, rather than the 3,000-row PyCaret leaderboard, compares all
candidate families under the same development protocol. Fixed-threshold metrics
use mean OOF probabilities across two repeats. All model parameters, environment
versions and fold fingerprints are retained alongside the metrics.


In [2]:
display(comparison[['model', 'cv_average_precision', 'cv_average_precision_std',
                    'cv_roc_auc', 'oof_poison_recall', 'oof_poison_precision',
                    'oof_f1', 'mean_fit_seconds']].round(6))
print('All candidates share fold fingerprint:', comparison.fold_fingerprint.nunique() == 1)


,model,cv_average_precision,cv_average_precision_std,cv_roc_auc,oof_poison_recall,oof_poison_precision,oof_f1,mean_fit_seconds
2,random_forest,0.783075,0.008699,0.833420,0.520792,0.853896,0.646986,0.520128
4,hist_gradient_boosting,0.758460,0.010047,0.812080,0.536634,0.815446,0.647293,0.220471
3,extra_trees,0.754268,0.012941,0.804034,0.482508,0.844111,0.614028,0.351862
1,logistic_regression,0.630651,0.010340,0.689445,0.345875,0.728790,0.469114,0.041113
0,majority_baseline,0.378750,0.000000,0.500000,0.000000,0.000000,0.000000,0.015551


All candidates share fold fingerprint: True


## Screening conclusion

In the executed checkpoint RF remains the strongest AP candidate. The two additional
families do not improve AP at these configurations. Boosting's fixed-threshold recall
can differ despite a lower AP: operating behavior and ranking quality are separate.
These are retained negative findings, not hidden experiments. RF and boosting are
reasonable candidates for a later bounded tuning study, subject to cost and error
analysis. Do not declare the final model or open the reserved test yet.
